# GEM TREQ: funciones de amplificación

Análisis exploratorio independiente de bronce. Fecha de revisión: 13 de noviembre de 2026.

Se examinan los archivos locales originales, no una tabla Delta ejecutada en Databricks.
No se limpia ni sobrescribe la fuente. Las transformaciones que aparezcan se hacen en copias
para evaluar lo que necesitará plata. Los resultados no prueban causalidad ni certifican seguridad.

Cada notebook sigue el orden: procedencia y unidad, calidad por columna y geometría,
distribuciones, límites y reglas para plata. No se muestran registros individuales de edificios oficiales.

In [ ]:
from pathlib import Path
import sys, json, re, unicodedata
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from IPython.display import display, HTML

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / "CLAUDE.md").exists() and (p / "data/externos").is_dir())
sys.path.insert(0, str(ROOT / "analisis_bronce"))
from soporte import (ficha, huella, leer_geojson, mostrar_perfil, geometria, conteos,
                     faltantes, guardar_resumen, grafico_barras, SALIDAS, CRS_METROS)
pd.set_option("display.max_columns", 24)
pd.set_option("display.max_rows", 60)
plt.rcParams.update({"font.size": 11, "figure.dpi": 110})

## 1. Procedencia y unidad

Una fila representa un código de sitio y un nivel de excitación de referencia. Son funciones modeladas, no observaciones del sismo.

In [ ]:
PATH=ROOT/"data/externos/gem_amplificacion_suelo.csv"
meta=ficha(PATH)
primera_linea=PATH.read_text().splitlines()[0]
print("Metadato previo a la cabecera:",primera_linea)
df=pd.read_csv(PATH,comment="#",dtype=str,keep_default_na=False)
calidad=mostrar_perfil(df)
x=df.copy()
for c in x.columns.drop("ampcode"):x[c]=pd.to_numeric(x[c],errors="coerce")
display(pd.Series(meta).to_frame("procedencia"))
resumen={"dataset":"11_gem_amplificacion",**meta,"filas":len(df),"calidad_columnas":calidad.to_dict("records"),"metadato":primera_linea}

## 2. Integridad de la matriz código × nivel × período

In [ ]:
conteo=x.groupby("ampcode")["level"].agg(filas="size",niveles="nunique",minimo="min",maximo="max")
display(conteo)
periodos=[c for c in x if c.startswith("SA(")]
niveles=sorted(x.level.unique())
combinaciones=pd.MultiIndex.from_product([sorted(x.ampcode.unique()),niveles],names=["ampcode","level"])
observadas=pd.MultiIndex.from_frame(x[["ampcode","level"]])
faltantes=combinaciones.difference(observadas)
qa={"codigos":int(x.ampcode.nunique()),"niveles":len(niveles),"periodos_sa":periodos,
    "duplicados_codigo_nivel":int(x.duplicated(["ampcode","level"]).sum()),"combinaciones_faltantes":len(faltantes),
    "factores_no_positivos":int(x[["PGA",*periodos]].le(0).sum().sum()),
    "sigmas_negativas":int(x.filter(like="sigma_").lt(0).sum().sum())}
display(pd.Series(qa).to_frame("control"))
display(x[["level","PGA",*periodos]].agg(["min","max"]))
resumen.update(qa)
resumen["codigos_lista"]=sorted(x.ampcode.unique().tolist())

## 3. Funciones al nivel de referencia que usa el pipeline

In [ ]:
nivel=0.2267
seleccion=x.loc[np.isclose(x.level,nivel)].set_index("ampcode")
assert len(seleccion)==x.ampcode.nunique(),"El nivel seleccionado no tiene una fila por código"
display(seleccion[["PGA",*periodos]])
fig,ax=plt.subplots(figsize=(10,5))
ts=[float(c[3:-1]) for c in periodos]
for z,row in seleccion.iterrows():ax.plot(ts,row[periodos].astype(float),marker="o",label=z)
ax.axhline(1,color="#444",linestyle="--",linewidth=.8)
ax.set(title="Amplificación modelada a nivel de roca 0,2267 g",xlabel="Período (s)",ylabel="Factor de amplificación")
ax.legend(title="Código",ncol=6,loc="upper center",bbox_to_anchor=(.5,-.18),fontsize=9);fig.tight_layout();fig.savefig(SALIDAS/"11_funciones.png",dpi=150);plt.show()
resumen["nivel_evaluado"]=nivel
resumen["factores_nivel_evaluado"]=seleccion[["PGA",*periodos]].reset_index().to_dict("records")

## Reglas decididas para plata

- **Incorporar como tabla de funciones**, con llave única `(ampcode, nivel_roca_g)`; pasar a formato largo por período en plata si facilita los cruces.
- Conservar la primera línea `vs30_ref=800` como metadato, no como fila. Mantener separados los factores y sus sigmas.
- Conservar PGA y SA con su definición; un factor puede ser menor que 1. No recortarlo artificialmente al intervalo 0–1.
- Elegir nivel y período es una hipótesis de modelado, no una medición del evento. Registrar el valor escogido y probar sensibilidad.
- Un código que no encuentra función debe producir una bandera y valor faltante, no amplificación 1 por defecto. La curva explícita `noaf` es un caso distinto.

In [ ]:
assert huella(PATH) == meta["sha256"], "La fuente cruda cambió durante el análisis"
guardar_resumen('11_gem_amplificacion', resumen)
print("Resumen y gráficos guardados; la fuente cruda conserva su SHA-256.")